In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.DataFrame({
    "factor_A": [
        "1",
        "1", "1",
        "2", "2",
        "2", "2",
        "3",
        "3", "3"
    ],
    "factor_B": [
        "1",
        "2", "2",
        "1", "1",
        "2", "2",
        "1",
        "2", "2"
    ],
    "y": [
        5,
        3, 4,
        6, 4,
        4, 3,
        7,
        6, 8
    ]
})

In [3]:
# 端点制約を課す
df_dummies = pd.get_dummies(df, columns = ["factor_A", "factor_B"], dtype = int, drop_first = True)

In [4]:
# 交互作用の追加
df_dummies["factor_A_2_B_2"] = df_dummies["factor_A_2"] * df_dummies["factor_B_2"] 
df_dummies["factor_A_3_B_2"] = df_dummies["factor_A_3"] * df_dummies["factor_B_2"]

In [5]:
df_dummies

,y,factor_A_2,factor_A_3,factor_B_2,factor_A_2_B_2,factor_A_3_B_2
0,5,0,0,0,0,0
1,3,0,0,1,0,0
2,4,0,0,1,0,0
3,6,1,0,0,0,0
4,4,1,0,0,0,0
5,4,1,0,1,1,0
6,3,1,0,1,1,0
7,7,0,1,0,0,0
8,6,0,1,1,0,1
9,8,0,1,1,0,1


# (a)

In [6]:
y = df_dummies["y"].values

# 交互作用項が0かどうかを検定
X_interaction = df_dummies.drop(["y"], axis = 1).values
X_interaction = np.column_stack([np.ones(len(y)), X_interaction])

# 交互作用項モデルの係数推定
beta_interaction = np.linalg.inv(X_interaction.T @ X_interaction) @ X_interaction.T @ y

# 交互作用項モデルの尺度つき逸脱度の計算
D_interaction = y @ y - beta_interaction.T @ X_interaction.T @ y

# 自由度
dof_interaction = X_interaction.shape[0] - X_interaction.shape[1]

In [7]:
# 加法モデル（交互作用なし）モデル
X_additive = df_dummies.drop(["y", "factor_A_2_B_2", "factor_A_3_B_2"], axis = 1).values
X_additive = np.column_stack([np.ones(len(y)), X_additive])

# 交互作用項モデルの係数推定
beta_additive = np.linalg.inv(X_additive.T @ X_additive) @ X_additive.T @ y

# 交互作用項モデルの尺度つき逸脱度の計算
D_additive = y @ y - beta_additive.T @ X_additive.T @ y

# 自由度
dof_additive = X_additive.shape[0] - X_additive.shape[1]

In [8]:
F = ((D_interaction-D_additive) / (dof_interaction - dof_additive)) / (D_interaction / dof_interaction)

# (b)

## (1)

In [52]:
# 因子Bのみモデル
X_B = df_dummies["factor_B_2"].values
X_B = np.column_stack([np.ones(len(y)), X_B])

# 因子Bのみモデルの係数推定
beta_B = np.linalg.inv(X_B.T @ X_B) @ X_B.T @ y

# 因子Bのみモデルの尺度つき逸脱度の計算
D_B = y @ y - beta_B.T @ X_B.T @ y

# 自由度
dof_B = X_B.shape[0] - X_B.shape[1]

# F値
F_B= ((D_B - D_additive) / (dof_B- dof_additive) ) / (D_interaction / dof_interaction)

In [54]:
dof_B- dof_additive, dof_interaction

(2, 4)

In [47]:
D_B - D_additive

np.float64(18.261904761904475)

In [48]:
F_B

np.float64(7.304761904762288)

In [49]:
# 因子Aのみモデル
X_A = df_dummies[["factor_A_2", "factor_A_3"]].values
X_A = np.column_stack([np.ones(len(y)), X_A])

# 因子Aのみモデルの係数推定
beta_A = np.linalg.inv(X_A.T @ X_A) @ X_A.T @ y

# 因子Aのみモデルの尺度つき逸脱度の計算
D_A = y @ y - beta_A.T @ X_A.T @ y

# 自由度
dof_A = X_A.shape[0] - X_A.shape[1]

# 最小モデル
X_M = np.column_stack([np.ones(len(y))])

# 最小モデルの係数推定
beta_M = np.linalg.inv(X_M.T @ X_M) @ X_M.T @ y

# 因子Bのみモデルの尺度つき逸脱度の計算
D_M = y @ y - beta_M.T @ X_M.T @ y

# 自由度
dof_M = X_M.shape[0] - X_M.shape[1]

# F値
F_M = ((D_M - D_A) / (dof_M - dof_A) ) / (D_interaction / dof_interaction)

In [50]:
(D_M - D_A)

np.float64(17.25)

In [51]:
F_M

np.float64(6.900000000000471)